# UC15 — Generación Automática de Resúmenes y Documentación

Convertir llamadas largas en resúmenes estructurados, notas CRM y emails post-llamada.

**Valor de negocio**: Ahorro de 5-8 minutos por llamada en documentación manual.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS RESUMENES_LLAMADAS_DB;
USE SCHEMA RESUMENES_LLAMADAS_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Transcripciones Largas (300 llamadas)

In [ ]:
CREATE OR REPLACE TABLE llamadas_completas AS
SELECT 'CALL' || LPAD(SEQ4(),5,'0') AS llamada_id,
    'AGT' || LPAD(MOD(SEQ4(),10)+1,3,'0') AS agente_id,
    'CLI' || LPAD(UNIFORM(1,500,RANDOM()),5,'0') AS cliente_id,
    DATEADD(day,-UNIFORM(1,60,RANDOM()),CURRENT_DATE()) AS fecha,
    UNIFORM(180,720,RANDOM()) AS duracion_seg,
    CASE MOD(SEQ4(),5)
        WHEN 0 THEN 'Agente: Buenos días, soy Carlos del departamento de siniestros. ¿En qué puedo ayudarle? Cliente: Hola Carlos, le llamo porque ayer tuve un accidente de coche. Iba por la M-30 y un coche se saltó un semáforo en rojo y me golpeó por el lateral derecho. Agente: Lamento mucho lo ocurrido. ¿Se encuentra bien? ¿Hubo heridos? Cliente: Yo estoy bien, solo un poco dolorido del cuello. El otro conductor también está bien. Agente: Me alegro de que no haya lesiones graves. Vamos a abrir el parte de siniestro. ¿Me puede dar su número de póliza? Cliente: Sí, es la póliza 12345-AUTO. Agente: Perfecto. Necesito algunos datos más: la matrícula del otro vehículo, si hay parte amistoso y si llamó a la policía. Cliente: La matrícula es 4567-BCD. Sí, rellenamos el parte amistoso. No llamamos a la policía porque no hubo heridos. Agente: Muy bien. Le voy a asignar un perito que contactará con usted en 24-48 horas. También le envío un email con el número de expediente y los próximos pasos. ¿Necesita vehículo de sustitución? Cliente: Sí, por favor, lo necesito para ir a trabajar. Agente: Perfecto, le gestiono un vehículo de sustitución por 5 días. ¿Alguna pregunta más? Cliente: No, muchas gracias Carlos. Agente: Gracias a usted. Le envío todo por email. Que se mejore.'
        WHEN 1 THEN 'Agente: Buenas tardes, soy Laura de atención al cliente. Cliente: Hola, llamo porque quiero cancelar mi seguro de hogar. Es demasiado caro y he encontrado algo mejor. Agente: Entiendo su preocupación. Antes de proceder, ¿me permite revisar su póliza para ver si hay alguna alternativa? Cliente: Bueno, pero llevo ya 3 años y cada año sube más. Agente: Veo que su prima actual es de 450€ anuales con cobertura completa. Tenemos una nueva tarifa con coberturas similares por 380€. Además, por su antigüedad, puedo aplicarle un 10% de descuento adicional. Cliente: Eso suena mejor. ¿Qué coberturas mantendría? Agente: Mantendría incendio, robo, daños por agua, RC y asistencia en hogar. La diferencia es en el límite de contenido que baja de 30.000€ a 25.000€. Cliente: Me parece bien. Vamos con esa opción. Agente: Perfecto, le envío la nueva propuesta por email para que la revise y firme digitalmente. ¿Necesita algo más? Cliente: No, gracias Laura. Agente: Gracias a usted por su confianza.'
        WHEN 2 THEN 'Agente: Buenos días, departamento de facturación. Cliente: Mire, le llamo muy enfadado. Me han cobrado dos veces el recibo de mi seguro de auto este mes. Son 600€ de más que necesito urgentemente. Agente: Lamento mucho la inconveniencia. Vamos a resolverlo inmediatamente. ¿Me puede dar su número de póliza? Cliente: Es la 78901-AUTO. Esto es la tercera vez que tengo problemas con los cobros. Agente: Entiendo su frustración y tiene toda la razón. Efectivamente veo un cobro duplicado. Voy a iniciar la devolución ahora mismo. El importe se reflejará en su cuenta en 3-5 días hábiles. Cliente: ¿No puede ser antes? Lo necesito para pagar facturas. Agente: Voy a marcar la devolución como urgente para que se procese en 24-48 horas. Además, como compensación por las molestias, le aplico un mes de bonificación en su próximo recibo. Cliente: Bueno, está bien. Pero que no vuelva a pasar. Agente: Tiene mi compromiso. Le envío un email de confirmación con el número de referencia de la devolución.'
        WHEN 3 THEN 'Agente: Hola, soy Miguel de asistencia. ¿En qué puedo ayudarle? Cliente: Buenas, necesito asistencia en carretera. Mi coche se ha quedado parado en la AP-7, cerca de Tarragona. No arranca y estoy en el arcén. Agente: ¿Se encuentra en un lugar seguro? Cliente: Sí, he puesto los triángulos y llevo el chaleco. Agente: Perfecto. Le localizo su posición. Veo que está en el kilómetro 142 de la AP-7 dirección Barcelona. Voy a enviar una grúa que llegará en aproximadamente 30-40 minutos. ¿Prefiere que le lleven al taller más cercano o a uno de confianza? Cliente: Al más cercano, por favor. Y otra cosa, ¿tengo vehículo de sustitución? Agente: Sí, su póliza incluye vehículo de sustitución. Una vez confirmada la avería, le gestionamos uno en el taller. También tiene hotel si necesita pernoctar. Cliente: No creo que haga falta el hotel. Gracias Miguel. Agente: La grúa va en camino. Le envío un SMS con los datos del conductor y la hora estimada de llegada.'
        ELSE 'Agente: Buenos días, le atiende Sofía. Cliente: Hola, tengo una duda sobre mi seguro de vida. He tenido un problema de salud y quiero saber si mi póliza cubre la baja laboral. Agente: Siento escuchar eso. ¿Puede darme su número de póliza para revisarla? Cliente: Es la 55667-VIDA. Agente: Veo que tiene una póliza de vida con cobertura de invalidez permanente y temporal. La baja laboral está cubierta si supera los 30 días consecutivos. ¿Cuánto tiempo lleva de baja? Cliente: Llevo 3 semanas, pero el médico dice que serán al menos 2 meses. Agente: En ese caso, cuando cumpla los 30 días puede solicitar la prestación. Necesitará el parte de baja y un informe médico. Le envío por email toda la documentación necesaria y los pasos a seguir. Cliente: ¿Y cuánto me pagarían? Agente: Su póliza cubre el 80% de su salario base declarado durante la baja. El pago se realiza mensualmente una vez aprobada la prestación. Cliente: Perfecto, muchas gracias Sofía. Agente: De nada. Espero que se recupere pronto. Aquí estamos para lo que necesite.'
    END AS transcripcion
FROM TABLE(GENERATOR(ROWCOUNT=>300));

SELECT llamada_id, agente_id, cliente_id, duracion_seg, LEFT(transcripcion,80) AS preview FROM llamadas_completas LIMIT 10;

---

## Paso 3: Generar Resúmenes Estructurados

In [ ]:
CREATE OR REPLACE TABLE resumenes AS
SELECT llamada_id, agente_id, cliente_id, fecha, duracion_seg,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Genera un resumen estructurado de esta llamada de seguros en formato JSON: {"motivo": "", "resumen": "máx 50 palabras", "acciones_realizadas": ["acción1","acción2"], "seguimiento_necesario": ["tarea1"], "sentimiento_cliente": "positivo/neutro/negativo", "productos_mencionados": ["producto1"]}.\n\nTranscripción: ' || transcripcion
    ) AS resumen_json
FROM llamadas_completas;

SELECT llamada_id, LEFT(resumen_json,400) AS resumen FROM resumenes LIMIT 5;

---

## Paso 4: Generar Notas CRM

In [ ]:
CREATE OR REPLACE TABLE notas_crm AS
SELECT llamada_id, agente_id, cliente_id, fecha,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Genera una nota breve para el CRM (máximo 3 líneas) a partir de esta llamada. Formato: MOTIVO: | ACCIÓN: | SEGUIMIENTO:. Transcripción: ' || transcripcion
    ) AS nota_crm
FROM llamadas_completas;

SELECT llamada_id, agente_id, cliente_id, nota_crm FROM notas_crm LIMIT 10;

---

## Paso 5: Generar Emails Post-Llamada

In [ ]:
CREATE OR REPLACE TABLE emails_postllamada AS
SELECT llamada_id, cliente_id, fecha,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Genera un email profesional post-llamada para el cliente de seguros. Debe ser breve (máx 100 palabras), cordial, resumir lo acordado y los próximos pasos. Firma como el agente. Transcripción: ' || transcripcion
    ) AS email_generado
FROM llamadas_completas;

SELECT llamada_id, cliente_id, LEFT(email_generado,300) AS email_preview FROM emails_postllamada LIMIT 5;

---

## Paso 6: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Generación Automática de Resúmenes')
st.markdown('### Documentación IA — Snowflake Cortex')

notas = session.table('notas_crm').to_pandas()
emails = session.table('emails_postllamada').to_pandas()

c1,c2,c3 = st.columns(3)
with c1: st.metric('Llamadas Procesadas', len(notas))
with c2: st.metric('Notas CRM Generadas', len(notas))
with c3: st.metric('Emails Generados', len(emails))

st.markdown('---')
st.subheader('Notas CRM Generadas')
st.dataframe(notas[['LLAMADA_ID','AGENTE_ID','CLIENTE_ID','FECHA','NOTA_CRM']].head(20), use_container_width=True, height=400)

st.markdown('---')
st.subheader('Preview de Email Post-Llamada')
sel = st.selectbox('Llamada', emails['LLAMADA_ID'].head(20).tolist())
email_sel = emails[emails['LLAMADA_ID']==sel]['EMAIL_GENERADO'].iloc[0]
st.text_area('Email:', value=email_sel, height=200)
st.caption('Desarrollado con Snowflake Cortex AI + Streamlit')

---

## Paso 7: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS RESUMENES_LLAMADAS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;